In [1]:
import io
import csv

In [2]:
def extract_raw_bytes():
    raw_csv_data = b"id,name,salary\n1,Alice,90000\n2,Bob,65000\n3,Charlie,110000"
    return io.BytesIO(raw_csv_data)

In [3]:
def transform_and_load(input_buffer):
    # 1. prepare output buffer
    output_buffer = io.BytesIO()

    # 2. decode the binary stream line-by-line
    text_reader = io.TextIOWrapper(input_buffer, encoding="utf-8")
    text_writer = io.TextIOWrapper(output_buffer, encoding="utf-8", write_through=True)

    csv_reader = csv.DictReader(text_reader)

    # define new target columns
    fieldnames = csv_reader.fieldnames + ["tax_owed"]
    csv_writer = csv.DictWriter(text_writer, fieldnames=fieldnames)
    csv_writer.writeheader()

    # 3. transform records row by row
    for row in csv_reader:
        salary = int(row.get("salary"))
        row["tax_owed"] = int(salary*0.2)
        print(row)
        csv_writer.writerow(row)

    # 4. rewind output memory stream so it's ready to be read/uploaded
    output_buffer.seek(0) 
    return output_buffer

In [5]:
input_stream = extract_raw_bytes()
output_stream = transform_and_load(input_stream)

{'id': '1', 'name': 'Alice', 'salary': '90000', 'tax_owed': 18000}
{'id': '2', 'name': 'Bob', 'salary': '65000', 'tax_owed': 13000}
{'id': '3', 'name': 'Charlie', 'salary': '110000', 'tax_owed': 22000}
